# REALISTA — Optional: Building the Stage-1/Stage-2 Dictionaries from Scratch

Both demo notebooks (`../demo_open_source_model.ipynb`, `../demo_reasoning_model.ipynb`) assume a
rephrasing dictionary (stage 1) and a latent-direction dictionary (stage 2) already exist for the
`(subject, question)` you want to attack. This notebook builds **both from scratch** for a single
unseen question.

**MMLU is used purely for concreteness.** Nothing in `dict_construction_utils.py` is MMLU-specific --
every function just takes a question, its choices, ground-truth index, and subject/topic string, so
the same calls work for a question from any other multiple-choice QA dataset.

Pipeline, matching the original paper (see `dict_construction_utils.py` for exact ports):

1. **Select concepts** from WordNet via a constrained optimization (`relaxed_select_autotune_RE`) --
   not an LLM's guess. Diversity + relevance + Qwen3-Embedding-8B / LLM-judged-editability floors.
2. **Generate rephrasings** per concept with an LLM (`semantic_equivalence_proposer`).
3. **Save the rephrasing dictionary** (`dictionary_utils.load_rephrasing_prompts` format).
4. **Extract latent perturbation directions** per rephrasing, in a local open-source model.
5. **Save the latent dictionary** (`dictionary_utils.load_latent_dict` format).
6. **Sanity-check** by reloading both with the same functions the two demos use.

One deviation from the paper: editability is scored only for the concepts most relevant to this
question, not the whole ~29k-concept pool -- the paper scores it once offline and reuses it across
every question, which a one-off demo can't amortize.

Run this notebook from inside this folder (or add it to `sys.path`).


In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import os
import sys

sys.path.append(os.path.abspath(".."))  # resolve src/'s flat modules (config.py, model_utils.py, ...)

import torch
from datasets import load_dataset

import config
import dictionary_utils
from config import MMLU_DATASET, LAYER_NUM_REGISTRY, GPT_5_MINI, FEASIBILITY_CHECKER_MODEL
from model_utils import GPT, load_model_and_tokenizer
from dictionary_utils import load_rephrasing_prompts, load_latent_dict, get_dictionary_and_base_latent
from arguments import RealistaArgs

from dict_construction_utils import (
    get_wordnet_adjective_concepts, load_embedding_model, embed_texts, select_concepts,
    build_rephrasing_dict_entry, save_rephrasing_dict,
    build_latent_dict_entry, save_latent_dict,
)

## 0. Resolve dictionary locations

Recomputes `REPHRASING_DICT_PATH`/`LATENT_DICT_PATH` as absolute paths (they default to relative,
which only resolves correctly when run from `src/`), and points `dictionary_utils` at them so the
sanity check below finds what this notebook saves.

In [2]:
REPO_ROOT = os.path.abspath("../..")


def _resolve(path_from_config, fallback_relative_to_repo_root):
    return path_from_config if os.path.isabs(path_from_config) else os.path.join(REPO_ROOT, fallback_relative_to_repo_root)


REPHRASING_DICT_DIR = _resolve(config.REPHRASING_DICT_PATH, "data/rephrasing_prompts")
LATENT_DICT_DIR = _resolve(config.LATENT_DICT_PATH, "data/latent_dict")

# dictionary_utils.py resolves REPHRASING_DICT_PATH/LATENT_DICT_PATH against
# config's own (possibly-relative) defaults, which only works when it's
# imported from src/ itself. Point it at the absolute paths above instead, so
# load_rephrasing_prompts/load_latent_dict (used in the sanity check below)
# find what this notebook just saved regardless of its own working directory.
dictionary_utils.REPHRASING_DICT_PATH = REPHRASING_DICT_DIR
dictionary_utils.LATENT_DICT_PATH = LATENT_DICT_DIR

## 1. Pick a question to build dictionaries for

Any `(subject, question_idx)` works -- saving always overwrites any existing dictionary entry for
that `(subject, question_idx)`, so re-running this notebook is safe. To use a different dataset,
replace `load_dataset(...)` below with however you source `question` / `choices` / `ground_truth_idx`
/ `subject`.

Dictionaries are saved under `DICT_SUBJECT = "demo_" + SUBJECT`, not `SUBJECT` itself, so anything
this notebook creates is clearly demo/test data and never collides with a real subject's dictionary
(`SUBJECT` is still used for `load_dataset(...)` and LLM prompt phrasing, since it must be a real MMLU
config name).

In [3]:
SUBJECT = "astronomy"
QUESTION_IDX = 1
DICT_SUBJECT = f"demo_{SUBJECT}"  # namespace for anything this notebook saves -- see section 1
MODEL_TYPE = "llama3_3b"       # local model used to compute latent perturbation directions
NUM_CONCEPTS = 5
NUM_REPHRASINGS_PER_CONCEPT = 3 # paper scale is ~100 concepts x 3 rephrasings; kept small for a fast demo

mmlu_dataset = load_dataset(MMLU_DATASET, SUBJECT)
cur_task_dict = mmlu_dataset["test"][QUESTION_IDX]

print(f"Question: {cur_task_dict['question']}")
for letter, choice in zip(["A", "B", "C", "D"], cur_task_dict["choices"]):
    tag = "  <-- GROUND TRUTH" if letter == "ABCD"[cur_task_dict["answer"]] else ""
    print(f"  {letter}. {choice}{tag}")

Question: If you know both the actual brightness of an object and its apparent brightness from your location then with no other information you can estimate:
  A. Its speed relative to you
  B. Its composition
  C. Its size
  D. Its distance from you  <-- GROUND TRUTH


In [4]:
rephrasing_path = os.path.join(REPHRASING_DICT_DIR, f"{DICT_SUBJECT}_rephrasings.json")

if os.path.isfile(rephrasing_path) and str(QUESTION_IDX) in json.load(open(rephrasing_path, encoding="utf-8")):
    print(f"subject={DICT_SUBJECT}, question_idx={QUESTION_IDX} already has a dictionary entry -- it will be overwritten.")
else:
    print(f"No existing dictionary entry for subject={DICT_SUBJECT}, question_idx={QUESTION_IDX}.")

No existing dictionary entry for subject=demo_astronomy, question_idx=1.


## 2. Stage 1a -- select concepts (WordNet + constrained optimization)

Concepts are adjective lemmas from WordNet, embedded with Qwen3-Embedding-8B -- not proposed by an
LLM. `select_concepts` picks a subset maximizing diversity subject to relevance/editability floors,
the same continuous-relaxation optimization as the paper's `relaxed_select_autotune_RE` (see
`dict_construction_utils.py`).

In [5]:
WORDNET_POOL_SIZE = 3000        # paper scale is ~29k WordNet adjective concepts; subsampled for speed
NUM_RELEVANCE_PREFILTER = 100    # candidates scored for editability before the constrained optimization

embedding_model = load_embedding_model()  # Qwen3-Embedding-8B
editability_llm = GPT(FEASIBILITY_CHECKER_MODEL, verbose=False)

concept_pool = get_wordnet_adjective_concepts(pool_size=WORDNET_POOL_SIZE, seed=0)
concept_pool_embeddings = embed_texts(concept_pool, embedding_model)

concepts = select_concepts(
    cur_task_dict["question"], concept_pool, concept_pool_embeddings, embedding_model,
    editability_llm, NUM_CONCEPTS, num_relevance_prefilter=NUM_RELEVANCE_PREFILTER,
)

2026-07-06 15:03:16.043489: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783364596.057174  838682 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783364596.061402  838682 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783364596.073009  838682 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783364596.073021  838682 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783364596.073023  838682 computation_placer.cc:177] computation placer alr

Scoring editability for the 100 most relevant candidate concepts...
Selected 5 concepts (relevance >= 1.82, editability >= 9.35):
  - precise  (relevance=0.351, editability=3)
  - calculable  (relevance=0.433, editability=2)
  - photometric  (relevance=0.493, editability=1)
  - grade-appropriate  (relevance=0.357, editability=2)
  - starry-eyed  (relevance=0.351, editability=2)


## 3. Stage 1b -- generate rephrasings per concept

Ask an LLM for `NUM_REPHRASINGS_PER_CONCEPT` semantically-equivalent rephrasings per concept.

In [6]:
concept_proposer_llm = GPT(GPT_5_MINI, verbose=False)

concept_to_rephrasings = build_rephrasing_dict_entry(
    cur_task_dict["question"], SUBJECT, cur_task_dict["answer"], cur_task_dict["choices"],
    concepts, NUM_REPHRASINGS_PER_CONCEPT, concept_proposer_llm,
)

Concept: precise
  - Given precise knowledge of an object's intrinsic luminosity and the flux measured at your location, what quantity can you deduce without any additional information?
  - Given a source's intrinsic luminosity and the flux measured at your location, what quantity can you determine precisely without additional data?
  - Given the intrinsic luminosity of an object and the flux you measure from Earth, what physical quantity can you determine precisely without additional data?
Concept: calculable
  - Knowing an object's intrinsic (absolute) luminosity and measuring how bright it appears to you, what parameter can you calculate about the object with no additional data?
  - If an object's intrinsic luminosity and the flux you measure from it are both known and no additional data are available, what calculable property of the object can you determine?
  - Given that an object's intrinsic luminosity and the flux you measure from it are known, what calculable property can you 

## 4. Save the rephrasing dictionary

In [7]:
saved_rephrasing_path = save_rephrasing_dict(REPHRASING_DICT_DIR, DICT_SUBJECT, QUESTION_IDX, concept_to_rephrasings)
print(f"Saved rephrasing dictionary: {os.path.basename(saved_rephrasing_path)}")

Saved rephrasing dictionary: demo_astronomy_rephrasings.json


## 5. Load the local target model

Used for stage 2 (latent perturbation directions), not concept embedding. Frees the embedding model's
GPU memory first -- both 8B and 3B+ models resident at once can exceed a single GPU. Any model in
`config.MODEL_REGISTRY` works; the latent dictionary is keyed by `model_type`.

In [8]:
del embedding_model
torch.cuda.empty_cache()

model, tokenizer = load_model_and_tokenizer(MODEL_TYPE)
layer_num = LAYER_NUM_REGISTRY[MODEL_TYPE]

Loading target LLM: llama3_3b (meta-llama/Llama-3.2-3B-Instruct)


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.44it/s]


## 6. Stage 2 -- extract latent perturbation directions

For every rephrasing, the difference between its hidden state and the original question's, at
`layer_num` (`config.LAYER_NUM_REGISTRY`).

In [9]:
concept_dict_entry = build_latent_dict_entry(
    cur_task_dict["question"], concept_to_rephrasings, model, tokenizer, layer_num,
)

Concept: precise  (3 perturbation direction(s))
Concept: calculable  (3 perturbation direction(s))
Concept: photometric  (3 perturbation direction(s))
Concept: grade-appropriate  (3 perturbation direction(s))
Concept: starry-eyed  (3 perturbation direction(s))


## 7. Save the latent dictionary

In [10]:
saved_latent_path = save_latent_dict(LATENT_DICT_DIR, MODEL_TYPE, DICT_SUBJECT, QUESTION_IDX, concept_dict_entry)
print(f"Saved latent dictionary: {MODEL_TYPE}/{DICT_SUBJECT}/{os.path.basename(saved_latent_path)}")

Saved latent dictionary: llama3_3b/demo_astronomy/llama3_3b_demo_astronomy_layer_0_latent_dictionary.pkl


## 8. Sanity check -- reload with the same functions the demos use

In [11]:
reloaded_rephrasings = load_rephrasing_prompts(DICT_SUBJECT)
assert str(QUESTION_IDX) in reloaded_rephrasings
print(f"Rephrasing dictionary OK -- {len(reloaded_rephrasings[str(QUESTION_IDX)])} concepts for question {QUESTION_IDX}.")

latent_dict = load_latent_dict(MODEL_TYPE, DICT_SUBJECT)
assert QUESTION_IDX in latent_dict
print(f"Latent dictionary OK -- {len(latent_dict[QUESTION_IDX])} concepts for question {QUESTION_IDX}.")

# mmlu_subject must be the real MMLU config name here (re-fetches the question via load_dataset);
# latent_dict itself was already loaded above from the DICT_SUBJECT namespace.
args = RealistaArgs(model_type=MODEL_TYPE, mmlu_subject=SUBJECT, mmlu_question_idx=QUESTION_IDX)
D_Z0, Z0, _, built_concepts = get_dictionary_and_base_latent(args, model, tokenizer, latent_dict)
print(f"D_Z0 shape: {D_Z0.shape}  (n_concepts={len(built_concepts)})")

Rephrasing dictionary OK -- 5 concepts for question 1.
Latent dictionary OK -- 5 concepts for question 1.


Padding perturbation directions: 100%|██████████| 5/5 [00:00<00:00, 250.99it/s]

D_Z0 shape: torch.Size([15, 39, 3072])  (n_concepts=15)


## Next steps

Both dictionaries now exist under `DICT_SUBJECT` for `QUESTION_IDX`. The two attack demos use a single
`mmlu_subject` for both `load_dataset(...)` and dictionary lookup, so `DICT_SUBJECT`'s `"demo_"` prefix
can't be plugged into them directly. To actually attack this question with `demo_open_source_model.ipynb`
or `demo_reasoning_model.ipynb`, rename the saved files to drop the `"demo_"` prefix -- i.e.
`data/rephrasing_prompts/DICT_SUBJECT_rephrasings.json` -> `.../SUBJECT_rephrasings.json`, and the
`.../MODEL_TYPE/DICT_SUBJECT/` latent-dict folder -> `.../MODEL_TYPE/SUBJECT/` -- then set
`mmlu_subject=SUBJECT` there.